
# Табличный редактор `df_out`

Интерактивный редактор на `ipywidgets` с пагинацией.

Возможности:

- поиск по УНП, клиенту и номеру договора;
- фильтры по валюте и типу операции;
- выбор отчетной даты;
- 10 / 25 / 50 строк на странице;
- редактирование сразу нескольких строк;
- редактирование `задолженность_{дата}`;
- автоматический пересчет `OD_{дата}` по валютному курсу из `df_rates`;
- редактирование НИ, ПФН, НВВ, рестры, обеспеченности, ГР и % резервирования;
- сохранение всей текущей страницы обратно в `df_out`;
- журнал ручных изменений `manual_edit_log`.

**Перед запуском должны существовать `df_out` и `df_rates`.**

Ожидаемая структура `df_rates`: валюты по строкам, отчетные даты по столбцам.  
Валюта может быть уже индексом либо находиться в обычном столбце `валюта` / `Валюта`.


In [ ]:

import re
import math
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output


## 1. Настройки и подготовка курсов

In [ ]:

# =============================================================================
# НАЗВАНИЯ ПОСТОЯННЫХ СТОЛБЦОВ В df_out
# Если у тебя они называются иначе — поменяй только эти значения.
# =============================================================================

COL_UNN = "УНП"
COL_CLIENT = "Наименование клиента"
COL_CONTRACT = "Номер договора"
COL_CURRENCY = "Валюта"
COL_OPERATION = "Тип операции"


# =============================================================================
# ПРОВЕРКИ
# =============================================================================

if "df_out" not in globals():
    raise NameError("Сначала должен быть создан dataframe df_out")

if "df_rates" not in globals():
    raise NameError("Сначала должен быть создан dataframe df_rates")

required_static_columns = [
    COL_UNN,
    COL_CLIENT,
    COL_CONTRACT,
    COL_CURRENCY,
    COL_OPERATION,
]

missing_static_columns = [
    col for col in required_static_columns
    if col not in df_out.columns
]

if missing_static_columns:
    raise ValueError(
        "В df_out отсутствуют обязательные столбцы: "
        + ", ".join(missing_static_columns)
    )


# =============================================================================
# ОТЧЕТНЫЕ ДАТЫ ИЗ САМОГО df_out
# =============================================================================

report_dates = []

for col in df_out.columns:
    match = re.match(
        r"^задолженность_(\d{2}\.\d{2}\.\d{4})$",
        str(col)
    )

    if match:
        report_dates.append(match.group(1))

report_dates = sorted(
    set(report_dates),
    key=lambda x: pd.to_datetime(x, format="%d.%m.%Y")
)

if not report_dates:
    raise ValueError(
        "В df_out не найдены столбцы вида "
        "'задолженность_01.01.2026'"
    )


# =============================================================================
# ПОДГОТОВКА КОПИИ ТАБЛИЦЫ КУРСОВ
# Оригинальный df_rates не меняем.
# =============================================================================

rates_table = df_rates.copy()

currency_col_in_rates = None

for col in rates_table.columns:
    if str(col).strip().lower() == "валюта":
        currency_col_in_rates = col
        break

if currency_col_in_rates is not None:
    rates_table[currency_col_in_rates] = (
        rates_table[currency_col_in_rates]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    rates_table = rates_table.set_index(
        currency_col_in_rates
    )

rates_table.index = (
    rates_table.index
    .astype(str)
    .str.strip()
    .str.upper()
)


# Приводим только те названия столбцов, которые можно распознать как даты.
rename_rates_columns = {}

for col in rates_table.columns:
    try:
        parsed = pd.to_datetime(
            col,
            dayfirst=True,
            errors="raise"
        )

        rename_rates_columns[col] = parsed.strftime(
            "%d.%m.%Y"
        )

    except Exception:
        pass

rates_table = rates_table.rename(
    columns=rename_rates_columns
)


def get_fx_rate(currency, date):
    currency = str(currency).strip().upper()

    if currency not in rates_table.index:
        return np.nan

    if date not in rates_table.columns:
        return np.nan

    value = rates_table.at[
        currency,
        date
    ]

    try:
        return float(value)
    except Exception:
        return np.nan


print("Отчетные даты:", report_dates)
print("Валют в таблице курсов:", len(rates_table.index))


## 2. Табличный редактор

In [ ]:

# =============================================================================
# ЖУРНАЛ РУЧНЫХ ИЗМЕНЕНИЙ
# =============================================================================

if "manual_edit_log" not in globals():
    manual_edit_log = pd.DataFrame(
        columns=[
            "index",
            "позиция строки",
            "УНП",
            "Номер договора",
            "дата",
            "поле",
            "старое значение",
            "новое значение",
        ]
    )


# =============================================================================
# СЛУЖЕБНЫЕ ФУНКЦИИ
# =============================================================================

def safe_float(value):
    try:
        if pd.isna(value):
            return 0.0
        return float(value)
    except Exception:
        return 0.0


def safe_binary(value):
    try:
        return 1 if float(value) == 1 else 0
    except Exception:
        return 0


def safe_gr(value):
    try:
        value = int(float(value))

        if value in [0, 1, 2, 3, 4, 5, 6]:
            return value

        return 0

    except Exception:
        return 0


def safe_text(value):
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.lower() in {
        "",
        "nan",
        "none",
        "0",
        "0.0",
    }:
        return ""

    return value


def ensure_object_column(column):
    if column not in df_out.columns:
        return

    if not pd.api.types.is_object_dtype(
        df_out[column].dtype
    ):
        df_out[column] = df_out[column].astype(
            object
        )


# =============================================================================
# СПИСКИ ФИЛЬТРОВ
# =============================================================================

currency_options = (
    ["Все"]
    +
    sorted(
        df_out[COL_CURRENCY]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )
)

operation_options = (
    ["Все"]
    +
    sorted(
        df_out[COL_OPERATION]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )
)


# =============================================================================
# ПАНЕЛЬ ПОИСКА И ФИЛЬТРОВ
# =============================================================================

search_unn = widgets.Text(
    placeholder="УНП...",
    description="УНП:",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "80px"},
)

search_client = widgets.Text(
    placeholder="Часть названия клиента...",
    description="Клиент:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "80px"},
)

search_contract = widgets.Text(
    placeholder="Номер договора...",
    description="Договор:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "80px"},
)

currency_filter = widgets.Dropdown(
    options=currency_options,
    value="Все",
    description="Валюта:",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "80px"},
)

operation_filter = widgets.Dropdown(
    options=operation_options,
    value="Все",
    description="Операция:",
    layout=widgets.Layout(width="360px"),
    style={"description_width": "80px"},
)

date_selector = widgets.Dropdown(
    options=report_dates,
    value=report_dates[0],
    description="Дата:",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "80px"},
)

page_size_selector = widgets.Dropdown(
    options=[10, 25, 50],
    value=25,
    description="Строк:",
    layout=widgets.Layout(width="190px"),
    style={"description_width": "70px"},
)

apply_filters_button = widgets.Button(
    description="Применить",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="150px"),
)

reset_filters_button = widgets.Button(
    description="Сбросить",
    icon="refresh",
    layout=widgets.Layout(width="140px"),
)


# =============================================================================
# ПАГИНАЦИЯ
# =============================================================================

prev_button = widgets.Button(
    description="← Назад",
    layout=widgets.Layout(width="120px"),
)

next_button = widgets.Button(
    description="Вперед →",
    layout=widgets.Layout(width="120px"),
)

page_info = widgets.HTML()

result_info = widgets.HTML()


# =============================================================================
# КНОПКА СОХРАНЕНИЯ
# =============================================================================

save_page_button = widgets.Button(
    description="Сохранить страницу в df_out",
    button_style="success",
    icon="save",
    layout=widgets.Layout(
        width="270px",
        height="42px",
    ),
)

reload_page_button = widgets.Button(
    description="Отменить несохраненные",
    icon="undo",
    layout=widgets.Layout(
        width="230px",
        height="42px",
    ),
)

status_output = widgets.Output()


# =============================================================================
# СОСТОЯНИЕ РЕДАКТОРА
# =============================================================================

editor_state = {
    "page": 0,
    "positions": np.arange(
        len(df_out),
        dtype=int
    ),
    "rows": [],
}


# =============================================================================
# ФИЛЬТРАЦИЯ
# =============================================================================

def get_filtered_positions():
    mask = np.ones(
        len(df_out),
        dtype=bool
    )

    unn = search_unn.value.strip().lower()

    if unn:
        mask &= (
            df_out[COL_UNN]
            .astype(str)
            .str.lower()
            .str.contains(
                unn,
                regex=False,
                na=False,
            )
            .to_numpy()
        )

    client = search_client.value.strip().lower()

    if client:
        mask &= (
            df_out[COL_CLIENT]
            .astype(str)
            .str.lower()
            .str.contains(
                client,
                regex=False,
                na=False,
            )
            .to_numpy()
        )

    contract = (
        search_contract.value
        .strip()
        .lower()
    )

    if contract:
        mask &= (
            df_out[COL_CONTRACT]
            .astype(str)
            .str.lower()
            .str.contains(
                contract,
                regex=False,
                na=False,
            )
            .to_numpy()
        )

    if currency_filter.value != "Все":
        mask &= (
            df_out[COL_CURRENCY]
            .astype(str)
            .str.strip()
            .eq(
                str(
                    currency_filter.value
                ).strip()
            )
            .to_numpy()
        )

    if operation_filter.value != "Все":
        mask &= (
            df_out[COL_OPERATION]
            .astype(str)
            .str.strip()
            .eq(
                str(
                    operation_filter.value
                ).strip()
            )
            .to_numpy()
        )

    return np.flatnonzero(
        mask
    )


# =============================================================================
# КОЛОНКИ ДЛЯ ВЫБРАННОЙ ДАТЫ
# =============================================================================

def get_date_columns():
    date = date_selector.value

    return {
        "debt": f"задолженность_{date}",
        "od": f"OD_{date}",
        "ni": f"НИ_{date}",
        "pfn": f"ПФН_{date}",
        "nvv": f"НВВ_{date}",
        "restra": f"рестра_{date}",
        "security": f"обеспеченность_{date}",
        "gr": f"ГР_{date}",
        "reserve": f"%рез_{date}",
    }


def validate_date_columns():
    columns = get_date_columns()

    missing = [
        col
        for col in columns.values()
        if col not in df_out.columns
    ]

    if missing:
        raise ValueError(
            "Для выбранной даты отсутствуют столбцы: "
            + ", ".join(missing)
        )


# =============================================================================
# ЗАГОЛОВКИ ТАБЛИЦЫ
# =============================================================================

HEADER_LAYOUT = [
    ("УНП", "115px"),
    ("Клиент", "260px"),
    ("Договор", "160px"),
    ("Валюта", "90px"),
    ("Операция", "170px"),
    ("Задолженность", "135px"),
    ("OD", "135px"),
    ("НИ", "70px"),
    ("ПФН", "70px"),
    ("НВВ", "70px"),
    ("Рестра", "145px"),
    ("Обеспеченность", "210px"),
    ("ГР", "75px"),
    ("% рез", "95px"),
]

GRID_COLUMNS = " ".join(
    width
    for _, width in HEADER_LAYOUT
)


def static_cell(value, width):
    value = "" if pd.isna(value) else str(value)

    return widgets.HTML(
        value=(
            "<div style='"
            "padding:6px;"
            "white-space:nowrap;"
            "overflow:hidden;"
            "text-overflow:ellipsis;"
            "border-bottom:1px solid #eee;"
            f"width:{width};"
            "'>"
            f"{value}"
            "</div>"
        ),
        layout=widgets.Layout(
            width=width
        ),
    )


def header_cell(title, width):
    return widgets.HTML(
        value=(
            "<div style='"
            "font-weight:600;"
            "padding:6px;"
            "border-bottom:2px solid #999;"
            "white-space:nowrap;"
            "'>"
            f"{title}"
            "</div>"
        ),
        layout=widgets.Layout(
            width=width
        ),
    )


# =============================================================================
# СОЗДАНИЕ ОДНОЙ РЕДАКТИРУЕМОЙ СТРОКИ
# =============================================================================

def create_editor_row(pos):
    date = date_selector.value
    columns = get_date_columns()

    row = df_out.iloc[
        pos
    ]

    currency = row[
        COL_CURRENCY
    ]

    debt_widget = widgets.FloatText(
        value=safe_float(
            row[
                columns["debt"]
            ]
        ),
        layout=widgets.Layout(
            width="135px"
        ),
    )

    od_widget = widgets.FloatText(
        value=safe_float(
            row[
                columns["od"]
            ]
        ),
        disabled=True,
        layout=widgets.Layout(
            width="135px"
        ),
    )

    ni_widget = widgets.Dropdown(
        options=[0, 1],
        value=safe_binary(
            row[
                columns["ni"]
            ]
        ),
        layout=widgets.Layout(
            width="70px"
        ),
    )

    pfn_widget = widgets.Dropdown(
        options=[0, 1],
        value=safe_binary(
            row[
                columns["pfn"]
            ]
        ),
        layout=widgets.Layout(
            width="70px"
        ),
    )

    nvv_widget = widgets.Dropdown(
        options=[0, 1],
        value=safe_binary(
            row[
                columns["nvv"]
            ]
        ),
        layout=widgets.Layout(
            width="70px"
        ),
    )

    restra_widget = widgets.Text(
        value=safe_text(
            row[
                columns["restra"]
            ]
        ),
        layout=widgets.Layout(
            width="145px"
        ),
    )

    security_widget = widgets.Text(
        value=safe_text(
            row[
                columns["security"]
            ]
        ),
        layout=widgets.Layout(
            width="210px"
        ),
    )

    gr_widget = widgets.Dropdown(
        options=[0, 1, 2, 3, 4, 5, 6],
        value=safe_gr(
            row[
                columns["gr"]
            ]
        ),
        layout=widgets.Layout(
            width="75px"
        ),
    )

    reserve_widget = widgets.FloatText(
        value=safe_float(
            row[
                columns["reserve"]
            ]
        ),
        layout=widgets.Layout(
            width="95px"
        ),
    )


    # ---------------------------------------------------------------------
    # OD ПЕРЕСЧИТЫВАЕТСЯ СРАЗУ ПРИ ИЗМЕНЕНИИ ЗАДОЛЖЕННОСТИ
    # ---------------------------------------------------------------------

    rate = get_fx_rate(
        currency,
        date
    )

    def debt_changed(change):
        if change["name"] != "value":
            return

        if pd.isna(rate):
            od_widget.value = 0.0

            with status_output:
                clear_output()
                print(
                    f"Не найден курс для валюты "
                    f"{currency} на {date}"
                )

            return

        od_widget.value = (
            safe_float(
                change["new"]
            )
            *
            rate
        )

    debt_widget.observe(
        debt_changed,
        names="value"
    )


    widgets_for_row = {
        "position": int(pos),
        "debt": debt_widget,
        "od": od_widget,
        "ni": ni_widget,
        "pfn": pfn_widget,
        "nvv": nvv_widget,
        "restra": restra_widget,
        "security": security_widget,
        "gr": gr_widget,
        "reserve": reserve_widget,
        "rate": rate,
    }

    cells = [
        static_cell(
            row[COL_UNN],
            "115px"
        ),
        static_cell(
            row[COL_CLIENT],
            "260px"
        ),
        static_cell(
            row[COL_CONTRACT],
            "160px"
        ),
        static_cell(
            row[COL_CURRENCY],
            "90px"
        ),
        static_cell(
            row[COL_OPERATION],
            "170px"
        ),
        debt_widget,
        od_widget,
        ni_widget,
        pfn_widget,
        nvv_widget,
        restra_widget,
        security_widget,
        gr_widget,
        reserve_widget,
    ]

    return widgets_for_row, cells


# =============================================================================
# ОБЛАСТЬ ТАБЛИЦЫ
# =============================================================================

table_container = widgets.Box(
    layout=widgets.Layout(
        width="100%",
        overflow_x="auto",
        border="1px solid #ddd",
    )
)


# =============================================================================
# ОТРИСОВКА ТЕКУЩЕЙ СТРАНИЦЫ
# =============================================================================

def render_page():
    validate_date_columns()

    positions = editor_state[
        "positions"
    ]

    page_size = int(
        page_size_selector.value
    )

    total_rows = len(
        positions
    )

    total_pages = max(
        1,
        math.ceil(
            total_rows
            /
            page_size
        )
    )

    editor_state["page"] = min(
        max(
            editor_state["page"],
            0
        ),
        total_pages - 1
    )

    page = editor_state[
        "page"
    ]

    start = (
        page
        *
        page_size
    )

    end = min(
        start + page_size,
        total_rows
    )

    current_positions = positions[
        start:end
    ]

    items = []

    for title, width in HEADER_LAYOUT:
        items.append(
            header_cell(
                title,
                width
            )
        )

    editor_rows = []

    for pos in current_positions:
        row_widgets, row_cells = (
            create_editor_row(
                int(pos)
            )
        )

        editor_rows.append(
            row_widgets
        )

        items.extend(
            row_cells
        )

    editor_state["rows"] = (
        editor_rows
    )

    grid = widgets.GridBox(
        children=items,
        layout=widgets.Layout(
            grid_template_columns=GRID_COLUMNS,
            grid_gap="2px 4px",
            align_items="center",
            width="max-content",
        )
    )

    table_container.children = [
        grid
    ]

    result_info.value = (
        f"<b>Найдено строк: {total_rows:,}</b>"
    )

    if total_rows == 0:
        result_info.value += (
            " — по текущим фильтрам ничего не найдено"
        )

    page_info.value = (
        f"<b>Страница "
        f"{page + 1} из {total_pages}</b>"
        f" &nbsp; | &nbsp; "
        f"строки "
        f"{start + 1 if total_rows else 0}"
        f"–{end}"
    )

    prev_button.disabled = (
        page <= 0
    )

    next_button.disabled = (
        page >= total_pages - 1
    )


# =============================================================================
# ПРИМЕНЕНИЕ ФИЛЬТРОВ
# =============================================================================

def apply_filters(button=None):
    editor_state["positions"] = (
        get_filtered_positions()
    )

    editor_state["page"] = 0

    with status_output:
        clear_output()

    render_page()


def reset_filters(button=None):
    search_unn.value = ""
    search_client.value = ""
    search_contract.value = ""

    currency_filter.value = "Все"
    operation_filter.value = "Все"

    editor_state["positions"] = np.arange(
        len(df_out),
        dtype=int
    )

    editor_state["page"] = 0

    render_page()


# =============================================================================
# ПЕРЕКЛЮЧЕНИЕ СТРАНИЦ
# =============================================================================

def previous_page(button=None):
    if editor_state["page"] > 0:
        editor_state["page"] -= 1
        render_page()


def next_page(button=None):
    editor_state["page"] += 1
    render_page()


# =============================================================================
# СОХРАНЕНИЕ ТЕКУЩЕЙ СТРАНИЦЫ
# =============================================================================

def save_current_page(button=None):
    global manual_edit_log

    date = date_selector.value
    columns = get_date_columns()

    # Текстовые столбцы заранее переводим в object,
    # чтобы гарантированно сохранялись новые значения.
    ensure_object_column(
        columns["restra"]
    )

    ensure_object_column(
        columns["security"]
    )

    new_log_rows = []
    missing_rates = []

    for row_widgets in editor_state[
        "rows"
    ]:
        pos = row_widgets[
            "position"
        ]

        real_index = df_out.index[
            pos
        ]

        rate = row_widgets[
            "rate"
        ]

        debt_value = safe_float(
            row_widgets[
                "debt"
            ].value
        )

        if pd.isna(rate):
            od_value = 0.0

            missing_rates.append(
                (
                    df_out.iloc[pos][
                        COL_CURRENCY
                    ],
                    date
                )
            )
        else:
            od_value = (
                debt_value
                *
                rate
            )

        # Еще раз синхронизируем отображаемый OD.
        row_widgets[
            "od"
        ].value = (
            od_value
        )

        new_values = {
            columns["debt"]:
                debt_value,

            columns["od"]:
                od_value,

            columns["ni"]:
                row_widgets[
                    "ni"
                ].value,

            columns["pfn"]:
                row_widgets[
                    "pfn"
                ].value,

            columns["nvv"]:
                row_widgets[
                    "nvv"
                ].value,

            columns["restra"]:
                (
                    str(
                        row_widgets[
                            "restra"
                        ].value
                    ).strip()
                    or 0
                ),

            columns["security"]:
                (
                    str(
                        row_widgets[
                            "security"
                        ].value
                    ).strip()
                    or 0
                ),

            columns["gr"]:
                row_widgets[
                    "gr"
                ].value,

            columns["reserve"]:
                row_widgets[
                    "reserve"
                ].value,
        }

        for column, new_value in new_values.items():
            col_pos = df_out.columns.get_loc(
                column
            )

            if not isinstance(
                col_pos,
                (int, np.integer)
            ):
                raise ValueError(
                    f"В df_out несколько столбцов "
                    f"с названием '{column}'"
                )

            old_value = df_out.iat[
                pos,
                col_pos
            ]

            df_out.iat[
                pos,
                col_pos
            ] = new_value

            saved_value = df_out.iat[
                pos,
                col_pos
            ]

            old_normalized = (
                ""
                if pd.isna(old_value)
                else str(old_value).strip()
            )

            new_normalized = (
                ""
                if pd.isna(saved_value)
                else str(saved_value).strip()
            )

            if (
                old_normalized
                !=
                new_normalized
            ):
                field_name = column

                suffix = (
                    f"_{date}"
                )

                if field_name.endswith(
                    suffix
                ):
                    field_name = field_name[
                        :-len(suffix)
                    ]

                new_log_rows.append({
                    "index":
                        real_index,

                    "позиция строки":
                        pos,

                    "УНП":
                        df_out.iloc[pos][
                            COL_UNN
                        ],

                    "Номер договора":
                        df_out.iloc[pos][
                            COL_CONTRACT
                        ],

                    "дата":
                        date,

                    "поле":
                        field_name,

                    "старое значение":
                        old_value,

                    "новое значение":
                        saved_value,
                })

    if new_log_rows:
        manual_edit_log = pd.concat(
            [
                manual_edit_log,
                pd.DataFrame(
                    new_log_rows
                ),
            ],
            ignore_index=True,
        )

    with status_output:
        clear_output()

        print(
            f"✓ Сохранено изменений: "
            f"{len(new_log_rows)}"
        )

        if missing_rates:
            unique_missing = sorted(
                set(
                    missing_rates
                )
            )

            print(
                "ВНИМАНИЕ: не найдены курсы:"
            )

            for currency, missing_date in unique_missing:
                print(
                    f"  {currency} — "
                    f"{missing_date}"
                )


# =============================================================================
# СОБЫТИЯ
# =============================================================================

apply_filters_button.on_click(
    apply_filters
)

reset_filters_button.on_click(
    reset_filters
)

prev_button.on_click(
    previous_page
)

next_button.on_click(
    next_page
)

save_page_button.on_click(
    save_current_page
)

reload_page_button.on_click(
    lambda button: render_page()
)

date_selector.observe(
    lambda change: (
        editor_state.update(
            {"page": 0}
        ),
        render_page()
    ),
    names="value",
)

page_size_selector.observe(
    lambda change: (
        editor_state.update(
            {"page": 0}
        ),
        render_page()
    ),
    names="value",
)


# =============================================================================
# ИНТЕРФЕЙС
# =============================================================================

title = widgets.HTML(
    "<h2>Табличный редактор df_out</h2>"
)

help_text = widgets.HTML(
    """
    <div style="margin-bottom:10px;">
    <b>Редактируются:</b>
    задолженность, НИ, ПФН, НВВ, рестра,
    обеспеченность, ГР и % резервирования.
    <br>
    <b>OD</b> пересчитывается автоматически:
    задолженность × курс валюты выбранной даты.
    </div>
    """
)

search_box = widgets.VBox([
    widgets.HBox([
        search_unn,
        search_contract,
    ]),
    search_client,
])

filter_box = widgets.HBox([
    currency_filter,
    operation_filter,
    date_selector,
    page_size_selector,
])

filter_buttons = widgets.HBox([
    apply_filters_button,
    reset_filters_button,
])

pager = widgets.HBox([
    prev_button,
    page_info,
    next_button,
])

save_box = widgets.HBox([
    save_page_button,
    reload_page_button,
])

editor_ui = widgets.VBox([
    title,
    help_text,
    search_box,
    filter_box,
    filter_buttons,
    result_info,
    pager,
    table_container,
    pager,
    save_box,
    status_output,
])

# Первичная загрузка.
editor_state["positions"] = np.arange(
    len(df_out),
    dtype=int
)

render_page()

display(
    editor_ui
)


## 3. Журнал ручных изменений

In [ ]:

display(manual_edit_log)


## 4. Проверка текущего `df_out`

In [ ]:

df_out.head()
